
# Fake News Detection - Pipeline Complet (Full Dataset)

Ce notebook exécute un pipeline complet de détection de Fake News sur **la totalité de la base de données**.
Parfait pour être exécuté sur Google Colab.

**Objectifs :**
- Fusionner deux datasets complets : Fake and Real News + LIAR (~57 000 articles).
- Tester exhaustivement : 2 Représentations (BoW, TF-IDF) × 3 Feature Selections (Chi2, MI, Fisher) × 8 Top_K × 3 Modèles ML.
- Modèles ML : Logistic Regression, Random Forest, et LinearSVC (optimisé pour les grands datasets).
- Tester 2 Modèles DL : LSTM (Architecture corrigée avec Global Average Pooling et vraie boucle d'entraînement) et DistilBERT.
- Générer des figures approfondies (Courbes d'apprentissage, Matrices de Confusion, ROC).

## 1. Importation des bibliothèques
    

In [ ]:

# Importation des bibliothèques de base
import pandas as pd
import numpy as np
import re
import string
import os
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss, roc_curve, confusion_matrix, ConfusionMatrixDisplay

# PyTorch et Transformers (Deep Learning)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import DistilBertTokenizer, DistilBertModel

# Création des dossiers de résultats si inexistants
os.makedirs("results/graphes", exist_ok=True)
print("Bibliothèques importées avec succès !")

# Paramétrage GPU pour PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device PyTorch utilisé : {device}")
    


## 2. Chargement de toutes les données
Nous fusionnons `Fake and Real News` et `LIAR` sans échantillonnage.
    

In [ ]:

def load_fake_real():
    print("Chargement Fake and Real News...")
    df_fake = pd.read_csv("data/fake_real/Fake.csv")
    df_true = pd.read_csv("data/fake_real/True.csv")
    df_fake['label'] = 1 # Fake
    df_true['label'] = 0 # Real
    df_fr = pd.concat([df_fake, df_true], ignore_index=True)
    df_fr = df_fr[['text', 'label']]
    return df_fr

def load_liar():
    print("Chargement LIAR Dataset...")
    cols = ['id', 'label_text', 'statement', 'subject', 'speaker', 'job', 'state', 'party', 
            'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context']
    
    df_train = pd.read_csv("data/liar/train.tsv", sep='\t', header=None, names=cols)
    df_valid = pd.read_csv("data/liar/valid.tsv", sep='\t', header=None, names=cols)
    df_test = pd.read_csv("data/liar/test.tsv", sep='\t', header=None, names=cols)
    
    df_liar = pd.concat([df_train, df_valid, df_test], ignore_index=True)
    
    fake_labels = ['pants-fire', 'false', 'barely-true']
    df_liar['label'] = df_liar['label_text'].apply(lambda x: 1 if x in fake_labels else 0)
    
    df_liar = df_liar.rename(columns={'statement': 'text'})
    df_liar = df_liar[['text', 'label']]
    return df_liar

df_fr = load_fake_real()
df_liar = load_liar()

# Concaténation totale
df_all = pd.concat([df_fr, df_liar], ignore_index=True)
df_all = df_all.dropna(subset=['text', 'label']).reset_index(drop=True)

# Mélange des données
df_sample = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Taille totale de la base de données : {len(df_sample)}")
print(df_sample['label'].value_counts())
print("\nAperçu des données :")
display(df_sample.head())
print("\nInformations :")
df_sample.info()
    


## 3. Prétraitement du Texte
    

In [ ]:

def clean_text(text):
    text = str(text).lower() 
    text = re.sub(r'https?://\S+|www\.\S+', '', text) 
    text = re.sub(r'<.*?>', '', text) 
    text = re.sub(r'\b(reuters)\b', '', text) 
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text) 
    text = re.sub(r'\s+', ' ', text).strip() 
    return text

tqdm.pandas(desc="Nettoyage du texte")
df_sample['clean_text'] = df_sample['text'].progress_apply(clean_text)

# Conversion robuste en NumPy
X = np.array(df_sample['clean_text'].astype(str).tolist(), dtype=object)
y = np.array(df_sample['label'].astype(int).tolist(), dtype=int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Shape de X_train : {X_train.shape}, Shape de X_test : {X_test.shape}")
display(df_sample[['text', 'clean_text']].head())
    


## 4. Extraction de Caractéristiques (BoW & TF-IDF)
On limite à 5000 features pour éviter des matrices RAM-intensives sur 57k lignes.
    

In [ ]:

print("Création du Bag of Words (BoW)...")
vectorizer_bow = CountVectorizer(max_features=5000, stop_words='english')
X_train_bow = vectorizer_bow.fit_transform(X_train)
X_test_bow = vectorizer_bow.transform(X_test)

print("Création de TF-IDF...")
vectorizer_tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer_tfidf.fit_transform(X_train)
X_test_tfidf = vectorizer_tfidf.transform(X_test)

print(f"Dimension BoW : {X_train_bow.shape}, TF-IDF : {X_train_tfidf.shape}")
    


## 5. Machine Learning Classique - Combinaisons Exhaustives
*Note : SVM a été remplacé par `LinearSVC` qui est un SVM optimisé pour les immenses matrices.*
    

In [ ]:

representations = {
    'BoW': (X_train_bow, X_test_bow),
    'TF-IDF': (X_train_tfidf, X_test_tfidf)
}

feature_selectors = {
    'Chi-square': chi2,
    'Mutual Information': mutual_info_classif,
    'Fisher': f_classif
}

top_ks = [20, 40, 100, 200, 400, 600, 1000, 1500]

models = {
    'Logistic Regression': LogisticRegression(max_iter=500, n_jobs=-1, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=42),
    'LinearSVC': LinearSVC(random_state=42, max_iter=1000) 
}

results_list = []
total_combinations = len(representations) * len(feature_selectors) * len(top_ks) * len(models)
print(f"Début de l'exécution des {total_combinations} combinaisons ML sur les {len(X_train)} données...")
counter = 0

best_ml_model_info = {'acc': 0, 'model': None, 'X_te_k': None, 'name': ''}

for rep_name, (X_tr, X_te) in representations.items():
    for fs_name, fs_func in feature_selectors.items():
        for k in top_ks:
            
            # Application FS
            selector = SelectKBest(score_func=fs_func, k=k)
            X_tr_k = selector.fit_transform(X_tr, y_train)
            X_te_k = selector.transform(X_te)
            
            for mod_name, model in models.items():
                counter += 1
                start_time = time.time()
                
                # Entraînement
                model.fit(X_tr_k, y_train)
                train_time = time.time() - start_time
                
                # Prédiction
                y_pred = model.predict(X_te_k)
                
                # Récupération probabilités ou distance (pour le ROC AUC)
                if hasattr(model, "predict_proba"):
                    y_prob = model.predict_proba(X_te_k)[:, 1]
                else:
                    # Pour LinearSVC, on utilise la decision_function et on normalise
                    y_dist = model.decision_function(X_te_k)
                    y_prob = 1 / (1 + np.exp(-y_dist)) # pseudo-probabilités via sigmoide
                
                # Évaluation
                acc = accuracy_score(y_test, y_pred)
                prec = precision_score(y_test, y_pred, zero_division=0)
                rec = recall_score(y_test, y_pred, zero_division=0)
                f1 = f1_score(y_test, y_pred, zero_division=0)
                auc = roc_auc_score(y_test, y_prob)
                
                # Log Loss (nécessite des probs entre 0 et 1)
                loss = log_loss(y_test, y_prob) if hasattr(model, "predict_proba") else np.nan
                
                # Sauvegarde du meilleur modèle pour la matrice de confusion plus tard
                if acc > best_ml_model_info['acc']:
                    best_ml_model_info = {'acc': acc, 'model': model, 'X_te_k': X_te_k, 'y_pred': y_pred, 'name': f"{mod_name} ({rep_name}, {fs_name} Top {k})"}

                results_list.append({
                    'Representation': rep_name,
                    'Feature_Selection': fs_name,
                    'Top_K': k,
                    'Model': mod_name,
                    'Accuracy': acc,
                    'Precision': prec,
                    'Recall': rec,
                    'F1_score': f1,
                    'ROC_AUC': auc,
                    'Log_Loss': loss,
                    'Training_Time': train_time
                })
                
                print(f"[{counter}/{total_combinations}] {rep_name} | {fs_name} (Top {k}) | {mod_name} -> Acc: {acc:.4f}, Temps: {train_time:.2f}s")
                
print(f"\nLe meilleur modèle ML est {best_ml_model_info['name']} avec Accuracy = {best_ml_model_info['acc']:.4f}")
    


## 6. Deep Learning - LSTM Corrigé
L'architecture a été corrigée : au lieu de prendre le dernier mot (qui était souvent du padding avec des zéros), le modèle effectue un **Global Average Pooling** sur toute la séquence pour en extraire le sens profond.
De plus, nous avons ajouté un tracking des courbes de Loss et Accuracy par Epoch !
    

In [ ]:

print("Préparation du LSTM...")

MAX_WORDS = 10000
MAX_LEN = 150
BATCH_SIZE = 64
EPOCHS = 10

# Vocabulaire
vocab = {}
word_index = 1
sequences_train = []
for text in X_train:
    seq = []
    for word in text.split()[:MAX_LEN]:
        if word not in vocab and len(vocab) < MAX_WORDS:
            vocab[word] = word_index
            word_index += 1
        if word in vocab:
            seq.append(vocab[word])
    sequences_train.append(seq)

sequences_test = []
for text in X_test:
    seq = []
    for word in text.split()[:MAX_LEN]:
        if word in vocab:
            seq.append(vocab[word])
    sequences_test.append(seq)

# Padding
def pad_sequences(seqs, max_len):
    padded = np.zeros((len(seqs), max_len), dtype=int)
    for i, seq in enumerate(seqs):
        length = min(len(seq), max_len)
        padded[i, :length] = seq[:length]
    return padded

X_train_lstm = pad_sequences(sequences_train, MAX_LEN)
X_test_lstm = pad_sequences(sequences_test, MAX_LEN)

train_data = TensorDataset(torch.tensor(X_train_lstm), torch.tensor(y_train, dtype=torch.float32))
test_data = TensorDataset(torch.tensor(X_test_lstm), torch.tensor(y_test, dtype=torch.float32))
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

# Modèle LSTM Corrigé (Global Average Pooling)
class ImprovedLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=64):
        super(ImprovedLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1) # Bidirectional = *2
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # x shape: [batch_size, seq_len]
        embedded = self.embedding(x)
        # lstm_out shape: [batch_size, seq_len, hidden_dim*2]
        lstm_out, _ = self.lstm(embedded)
        
        # CORRECTION : Global Average Pooling sur la longueur de séquence (dim=1)
        # On évite de prendre le dernier état rempli de zéros !
        pooled_out = lstm_out.mean(dim=1) 
        
        out = self.fc(pooled_out)
        return self.sigmoid(out).squeeze()

model_lstm = ImprovedLSTM(len(vocab) + 1).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.001)

# Entraînement avec suivi des courbes
train_losses, val_losses, val_accuracies = [], [], []

start_time = time.time()
for epoch in range(EPOCHS):
    model_lstm.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_lstm(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation
    model_lstm.eval()
    val_loss = 0.0
    val_preds, val_trues = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_lstm(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            val_preds.extend((outputs.cpu().numpy() >= 0.5).astype(int))
            val_trues.extend(labels.cpu().numpy())
            
    avg_val_loss = val_loss / len(test_loader)
    val_acc = accuracy_score(val_trues, val_preds)
    val_losses.append(avg_val_loss)
    val_accuracies.append(val_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS} -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

lstm_time = time.time() - start_time

# Extraction probabilités finales test
model_lstm.eval()
y_pred_probs_lstm = []
with torch.no_grad():
    for inputs, _ in test_loader:
        inputs = inputs.to(device)
        outputs = model_lstm(inputs)
        y_pred_probs_lstm.extend(outputs.cpu().numpy())

y_pred_probs_lstm = np.array(y_pred_probs_lstm)
y_pred_lstm = (y_pred_probs_lstm >= 0.5).astype(int)

# Sauvegarde LSTM
results_list.append({
    'Representation': 'Word Embeddings',
    'Feature_Selection': 'None',
    'Top_K': 'All',
    'Model': 'LSTM',
    'Accuracy': accuracy_score(y_test, y_pred_lstm),
    'Precision': precision_score(y_test, y_pred_lstm, zero_division=0),
    'Recall': recall_score(y_test, y_pred_lstm, zero_division=0),
    'F1_score': f1_score(y_test, y_pred_lstm, zero_division=0),
    'ROC_AUC': roc_auc_score(y_test, y_pred_probs_lstm),
    'Log_Loss': log_loss(y_test, y_pred_probs_lstm),
    'Training_Time': lstm_time
})
print(f"\nModèle LSTM terminé -> Acc Finale: {accuracy_score(y_test, y_pred_lstm):.4f}, Temps: {lstm_time:.2f}s")
    


## 7. Deep Learning - DistilBERT
Extraction d'embeddings avec Transformers, suivie d'une régression logistique.
    

In [ ]:

print("Préparation de DistilBERT...")
tokenizer_bert = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model_bert = DistilBertModel.from_pretrained('distilbert-base-uncased').to(device)
model_bert.eval() 

# Batching pour extraction GPU/CPU
def get_bert_embeddings(texts, batch_size=128):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Extraction BERT"):
        batch_texts = texts[i:i+batch_size].tolist()
        inputs = tokenizer_bert(batch_texts, return_tensors='pt', truncation=True, padding=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model_bert(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(cls_embeddings)
    return np.vstack(embeddings)

start_time = time.time()
print("Extraction Train embeddings...")
X_train_bert = get_bert_embeddings(X_train, batch_size=128)
print("Extraction Test embeddings...")
X_test_bert = get_bert_embeddings(X_test, batch_size=128)

model_bert_lr = LogisticRegression(max_iter=500, n_jobs=-1)
model_bert_lr.fit(X_train_bert, y_train)
bert_time = time.time() - start_time

y_pred_bert = model_bert_lr.predict(X_test_bert)
y_prob_bert = model_bert_lr.predict_proba(X_test_bert)[:, 1]

results_list.append({
    'Representation': 'DistilBERT',
    'Feature_Selection': 'None',
    'Top_K': 'All',
    'Model': 'BERT + LR',
    'Accuracy': accuracy_score(y_test, y_pred_bert),
    'Precision': precision_score(y_test, y_pred_bert, zero_division=0),
    'Recall': recall_score(y_test, y_pred_bert, zero_division=0),
    'F1_score': f1_score(y_test, y_pred_bert, zero_division=0),
    'ROC_AUC': roc_auc_score(y_test, y_prob_bert),
    'Log_Loss': log_loss(y_test, y_prob_bert),
    'Training_Time': bert_time
})
print(f"Modèle DistilBERT+LR terminé -> Acc: {accuracy_score(y_test, y_pred_bert):.4f}, Temps: {bert_time:.2f}s")
    


## 8. Sauvegarde, Visualisation et Métriques Avancées (Graphes)
    

In [ ]:

df_results = pd.DataFrame(results_list)
df_results.to_csv("results/perfs_modeles_complete.csv", index=False)
print("Résultats complets sauvegardés.")

display(df_results.sort_values(by="Accuracy", ascending=False).head(10))

# ----- GRAPHE 1 : LSTM Learning Curves -----
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS+1), train_losses, label='Train Loss', marker='o')
plt.plot(range(1, EPOCHS+1), val_losses, label='Validation Loss', marker='o')
plt.title("LSTM - Loss par Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS+1), val_accuracies, label='Validation Accuracy', marker='o', color='green')
plt.title("LSTM - Accuracy par Epoch")
plt.legend()
plt.tight_layout()
plt.savefig("results/graphes/lstm_learning_curves.png")
plt.show()

# ----- GRAPHE 2 : Matrices de Confusion -----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Meilleur Modèle ML
cm_ml = confusion_matrix(y_test, best_ml_model_info['y_pred'])
disp_ml = ConfusionMatrixDisplay(confusion_matrix=cm_ml, display_labels=['Real (0)', 'Fake (1)'])
disp_ml.plot(ax=axes[0], cmap='Blues')
axes[0].set_title(f"Meilleur ML: {best_ml_model_info['name']}")

# 2. LSTM
cm_lstm = confusion_matrix(y_test, y_pred_lstm)
disp_lstm = ConfusionMatrixDisplay(confusion_matrix=cm_lstm, display_labels=['Real (0)', 'Fake (1)'])
disp_lstm.plot(ax=axes[1], cmap='Greens')
axes[1].set_title("Deep Learning: LSTM")

# 3. DistilBERT
cm_bert = confusion_matrix(y_test, y_pred_bert)
disp_bert = ConfusionMatrixDisplay(confusion_matrix=cm_bert, display_labels=['Real (0)', 'Fake (1)'])
disp_bert.plot(ax=axes[2], cmap='Purples')
axes[2].set_title("Deep Learning: DistilBERT")

plt.tight_layout()
plt.savefig("results/graphes/confusion_matrices.png")
plt.show()

# ----- GRAPHE 3 : Courbes ROC des modèles DL -----
plt.figure(figsize=(8, 6))

# LSTM
fpr_lstm, tpr_lstm, _ = roc_curve(y_test, y_pred_probs_lstm)
auc_lstm = roc_auc_score(y_test, y_pred_probs_lstm)
plt.plot(fpr_lstm, tpr_lstm, label=f'LSTM (AUC = {auc_lstm:.4f})')

# BERT
fpr_bert, tpr_bert, _ = roc_curve(y_test, y_prob_bert)
auc_bert = roc_auc_score(y_test, y_prob_bert)
plt.plot(fpr_bert, tpr_bert, label=f'DistilBERT (AUC = {auc_bert:.4f})')

plt.plot([0, 1], [0, 1], 'k--', label='Aléatoire (AUC = 0.5)')
plt.title("Courbes ROC - Modèles Deep Learning")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.savefig("results/graphes/roc_dl_models.png")
plt.show()
    